# Merge das bases raw

Ler todos os arquivos `.parquet` em `../data/raw` e consolida em uma base mensal única.

In [16]:
from pathlib import Path
from functools import reduce

import pandas as pd

In [17]:
RAW_DIR = Path("../data/raw")
parquet_files = sorted(RAW_DIR.glob("*.parquet"))

print(f"Arquivos encontrados: {len(parquet_files)}")
for p in parquet_files:
    print(" -", p.name)

Arquivos encontrados: 11
 - cambio_ptax_venda.parquet
 - ibc_br_dessaz.parquet
 - inadimplencia_carteira_total.parquet
 - inadimplencia_pf_total.parquet
 - ipca_variacao_mensal.parquet
 - pmc_volume_vendas_dessazonalizado.parquet
 - rendimento_medio_real.parquet
 - saldo_carteira_credito_total.parquet
 - selic_acumulada_mes.parquet
 - taxa_desocupacao.parquet
 - taxa_juros_pf_total.parquet


In [18]:
# Range de datas de cada arquivo antes do merge
for path in parquet_files:
    df = pd.read_parquet(path).copy()

    if "data" in df.columns:
        serie_data = pd.to_datetime(df["data"], errors="coerce")
    elif isinstance(df.index, pd.DatetimeIndex):
        serie_data = pd.to_datetime(df.index, errors="coerce")
    else:
        candidatos = [c for c in df.columns if "data" in c.lower() or "date" in c.lower()]
        if candidatos:
            serie_data = pd.to_datetime(df[candidatos[0]], errors="coerce")
        else:
            print(f"{path.name}: sem coluna de data identificÃ¡vel")
            continue

    serie_data = serie_data.dropna()
    if serie_data.empty:
        print(f"{path.name}: sem datas vÃ¡lidas")
    else:
        print(f"{path.name}: {serie_data.min().date()} -> {serie_data.max().date()}")

cambio_ptax_venda.parquet: 2011-01-03 -> 2026-02-27
ibc_br_dessaz.parquet: 2011-01-01 -> 2025-12-01
inadimplencia_carteira_total.parquet: 2011-03-01 -> 2026-01-01
inadimplencia_pf_total.parquet: 2011-03-01 -> 2026-01-01
ipca_variacao_mensal.parquet: 2011-01-01 -> 2026-01-01
pmc_volume_vendas_dessazonalizado.parquet: 2000-01-01 -> 2025-12-01
rendimento_medio_real.parquet: 2012-04-01 -> 2025-12-01
saldo_carteira_credito_total.parquet: 2011-01-01 -> 2026-01-01
selic_acumulada_mes.parquet: 2011-01-01 -> 2026-02-01
taxa_desocupacao.parquet: 2012-04-01 -> 2025-12-01
taxa_juros_pf_total.parquet: 2011-03-01 -> 2026-01-01


In [ ]:
def preparar_base_mensal(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()

    if "data" not in df.columns:
        if isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index().rename(columns={df.index.name or "index": "data"})
        else:
            candidatos = [c for c in df.columns if "data" in c.lower() or "date" in c.lower()]
            if not candidatos:
                raise ValueError(f"Nenhuma coluna de data encontrada em {path.name}")
            df = df.rename(columns={candidatos[0]: "data"})

    df["data"] = pd.to_datetime(df["data"], errors="coerce")
    df = df.dropna(subset=["data"])

    # Chave mensal de referencia
    df["data_mes"] = df["data"].dt.to_period("M").dt.to_timestamp()

    # Agrega para frequência mensal caso exista mais de um registro no mes
    colunas_numericas = [c for c in df.select_dtypes(include="number").columns if c not in {"data"}]
    if colunas_numericas:
        base = df.groupby("data_mes", as_index=False)[colunas_numericas].mean()
    else:
        base = df[["data_mes"]].drop_duplicates()


    return base

In [20]:
bases_mensais = [preparar_base_mensal(path) for path in parquet_files]

base_unica = reduce(
    lambda esquerda, direita: esquerda.merge(direita, on="data_mes", how="outer"),
    bases_mensais,
)

base_unica = base_unica.sort_values("data_mes").reset_index(drop=True)

print(f"Shape final: {base_unica.shape}")
base_unica.head()

Shape final: (314, 12)


,data_mes,cambio_ptax_venda,ibc_br_dessaz,inadimplencia_carteira_total,inadimplencia_pf_total,ipca_variacao_mensal,pmc_volume_vendas_dessazonalizado,rendimento_medio_real,saldo_carteira_credito_total,selic_acumulada_mes,taxa_desocupacao,taxa_juros_pf_total
0,2000-01-01,NaN,NaN,NaN,NaN,NaN,5013569.0,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,NaN,NaN,NaN,NaN,NaN,5060764.0,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,NaN,NaN,NaN,NaN,NaN,5089459.0,NaN,NaN,NaN,NaN,NaN
3,2000-04-01,NaN,NaN,NaN,NaN,NaN,5096019.0,NaN,NaN,NaN,NaN,NaN
4,2000-05-01,NaN,NaN,NaN,NaN,NaN,5126968.0,NaN,NaN,NaN,NaN,NaN


In [21]:
# Mantem apenas o range mensal comum entre todas as bases
ranges = []
for base in bases_mensais:
    ranges.append((base["data_mes"].min(), base["data_mes"].max()))

inicio_comum = max(inicio for inicio, _ in ranges)
fim_comum = min(fim for _, fim in ranges)

if inicio_comum > fim_comum:
    raise ValueError("Nao existe intersecao mensal comum entre todas as bases.")

df_new = base_unica.loc[
    (base_unica["data_mes"] >= inicio_comum) & (base_unica["data_mes"] <= fim_comum)
].sort_values("data_mes").reset_index(drop=True)

print(f"Range comum: {inicio_comum.date()} -> {fim_comum.date()}")
print(f"Shape (range comum): {df_new.shape}")
df_new.head()

Range comum: 2012-04-01 -> 2025-12-01
Shape (range comum): (165, 12)


,data_mes,cambio_ptax_venda,ibc_br_dessaz,inadimplencia_carteira_total,inadimplencia_pf_total,ipca_variacao_mensal,pmc_volume_vendas_dessazonalizado,rendimento_medio_real,saldo_carteira_credito_total,selic_acumulada_mes,taxa_desocupacao,taxa_juros_pf_total
0,2012-04-01,1.854835,98.87951,3.76,5.41,0.64,9387725.0,3075.0,2106585.0,0.71,78.0,34.30
1,2012-05-01,1.985991,100.02006,3.76,5.51,0.36,9368144.0,3067.0,2142061.0,0.74,77.0,32.34
2,2012-06-01,2.049195,100.74394,3.71,5.43,0.08,9517799.0,3073.0,2175135.0,0.64,76.0,31.58
3,2012-07-01,2.028736,100.90465,3.72,5.42,0.43,9572403.0,3089.0,2192888.0,0.68,75.0,30.71
4,2012-08-01,2.029443,101.24114,3.75,5.40,0.41,9584401.0,3099.0,2219866.0,0.69,73.0,29.84


In [22]:
df_new.isnull().sum()

data_mes                             0
cambio_ptax_venda                    0
ibc_br_dessaz                        0
inadimplencia_carteira_total         0
inadimplencia_pf_total               0
ipca_variacao_mensal                 0
pmc_volume_vendas_dessazonalizado    0
rendimento_medio_real                0
saldo_carteira_credito_total         0
selic_acumulada_mes                  0
taxa_desocupacao                     0
taxa_juros_pf_total                  0
dtype: int64

In [23]:
# salvar a base consolidada
df_new.to_parquet("../data/processed/base_unica_mensal.parquet", index=False)

# Feature Engineering

In [1]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_parquet("../data/processed/base_unica_mensal.parquet")
print(df.shape)

(165, 12)


In [25]:
df.head()

,data_mes,cambio_ptax_venda,ibc_br_dessaz,inadimplencia_carteira_total,inadimplencia_pf_total,ipca_variacao_mensal,pmc_volume_vendas_dessazonalizado,rendimento_medio_real,saldo_carteira_credito_total,selic_acumulada_mes,taxa_desocupacao,taxa_juros_pf_total
0,2012-04-01,1.854835,98.87951,3.76,5.41,0.64,9387725.0,3075.0,2106585.0,0.71,78.0,34.30
1,2012-05-01,1.985991,100.02006,3.76,5.51,0.36,9368144.0,3067.0,2142061.0,0.74,77.0,32.34
2,2012-06-01,2.049195,100.74394,3.71,5.43,0.08,9517799.0,3073.0,2175135.0,0.64,76.0,31.58
3,2012-07-01,2.028736,100.90465,3.72,5.42,0.43,9572403.0,3089.0,2192888.0,0.68,75.0,30.71
4,2012-08-01,2.029443,101.24114,3.75,5.40,0.41,9584401.0,3099.0,2219866.0,0.69,73.0,29.84


In [26]:
df.dtypes

data_mes                             datetime64[ns]
cambio_ptax_venda                           float64
ibc_br_dessaz                               float64
inadimplencia_carteira_total                float64
inadimplencia_pf_total                      float64
ipca_variacao_mensal                        float64
pmc_volume_vendas_dessazonalizado           float64
rendimento_medio_real                       float64
saldo_carteira_credito_total                float64
selic_acumulada_mes                         float64
taxa_desocupacao                            float64
taxa_juros_pf_total                         float64
dtype: object

In [3]:
# Garantir ordenação e índice temporal
df["data_mes"] = pd.to_datetime(df["data_mes"])
df = df.sort_values("data_mes").reset_index(drop=True)
df = df.set_index("data_mes")
df.index.name = "data_mes"

# Checagens básicas
display(df.head())
print("Período:", df.index.min().date(), "->", df.index.max().date())
print("Frequência inferida:", pd.infer_freq(df.index))

# Missing values
na = df.isna().sum().sort_values(ascending=False)
display(na[na > 0].to_frame("n_missing"))

,cambio_ptax_venda,ibc_br_dessaz,inadimplencia_carteira_total,inadimplencia_pf_total,ipca_variacao_mensal,pmc_volume_vendas_dessazonalizado,rendimento_medio_real,saldo_carteira_credito_total,selic_acumulada_mes,taxa_desocupacao,taxa_juros_pf_total
data_mes,,,,,,,,,,,
2012-04-01,1.854835,98.87951,3.76,5.41,0.64,9387725.0,3075.0,2106585.0,0.71,78.0,34.30
2012-05-01,1.985991,100.02006,3.76,5.51,0.36,9368144.0,3067.0,2142061.0,0.74,77.0,32.34
2012-06-01,2.049195,100.74394,3.71,5.43,0.08,9517799.0,3073.0,2175135.0,0.64,76.0,31.58
2012-07-01,2.028736,100.90465,3.72,5.42,0.43,9572403.0,3089.0,2192888.0,0.68,75.0,30.71
2012-08-01,2.029443,101.24114,3.75,5.40,0.41,9584401.0,3099.0,2219866.0,0.69,73.0,29.84


Período: 2012-04-01 -> 2025-12-01
Frequência inferida: MS


,n_missing


## Criando lags temporais

In [4]:
TARGET = "inadimplencia_pf_total"

EXOG = [
    "cambio_ptax_venda",
    "ibc_br_dessaz",
    "inadimplencia_carteira_total",
    "ipca_variacao_mensal",
    "pmc_volume_vendas_dessazonalizado",
    "rendimento_medio_real",
    "saldo_carteira_credito_total",
    "selic_acumulada_mes",
    "taxa_desocupacao",
    "taxa_juros_pf_total",
]

# Lags que vamos usar (começando leve; depois a gente pode ampliar)
LAGS = [1, 3, 6, 12]

df_fe = df.copy()

# 1) Lags do alvo (autorregressivos)
for k in LAGS:
    df_fe[f"{TARGET}__lag{k}"] = df_fe[TARGET].shift(k)

# 2) Lags das exógenas
for col in EXOG:
    for k in LAGS:
        df_fe[f"{col}__lag{k}"] = df_fe[col].shift(k)

# 3) Separar X e y (por enquanto com tudo; vamos refinar depois)
feature_cols = [c for c in df_fe.columns if c.endswith(tuple([f"lag{k}" for k in LAGS]))]
X = df_fe[feature_cols]
y = df_fe[TARGET]

print("Target:", TARGET)
print("N features:", X.shape[1])
display(df_fe[[TARGET] + feature_cols].head(15))

# 4) Quantos rows vão ser perdidos por causa dos lags
print("Rows com qualquer NA em X:", X.isna().any(axis=1).sum())
print("Rows totais:", len(df_fe))

Target: inadimplencia_pf_total
N features: 44


,inadimplencia_pf_total,inadimplencia_pf_total__lag1,inadimplencia_pf_total__lag3,inadimplencia_pf_total__lag6,inadimplencia_pf_total__lag12,cambio_ptax_venda__lag1,cambio_ptax_venda__lag3,cambio_ptax_venda__lag6,cambio_ptax_venda__lag12,ibc_br_dessaz__lag1,ibc_br_dessaz__lag3,ibc_br_dessaz__lag6,ibc_br_dessaz__lag12,inadimplencia_carteira_total__lag1,inadimplencia_carteira_total__lag3,inadimplencia_carteira_total__lag6,inadimplencia_carteira_total__lag12,ipca_variacao_mensal__lag1,ipca_variacao_mensal__lag3,ipca_variacao_mensal__lag6,ipca_variacao_mensal__lag12,pmc_volume_vendas_dessazonalizado__lag1,pmc_volume_vendas_dessazonalizado__lag3,pmc_volume_vendas_dessazonalizado__lag6,pmc_volume_vendas_dessazonalizado__lag12,rendimento_medio_real__lag1,rendimento_medio_real__lag3,rendimento_medio_real__lag6,rendimento_medio_real__lag12,saldo_carteira_credito_total__lag1,saldo_carteira_credito_total__lag3,saldo_carteira_credito_total__lag6,saldo_carteira_credito_total__lag12,selic_acumulada_mes__lag1,selic_acumulada_mes__lag3,selic_acumulada_mes__lag6,selic_acumulada_mes__lag12,taxa_desocupacao__lag1,taxa_desocupacao__lag3,taxa_desocupacao__lag6,taxa_desocupacao__lag12,taxa_juros_pf_total__lag1,taxa_juros_pf_total__lag3,taxa_juros_pf_total__lag6,taxa_juros_pf_total__lag12
data_mes,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2012-04-01,5.41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-05-01,5.51,5.41,NaN,NaN,NaN,1.854835,NaN,NaN,NaN,98.87951,NaN,NaN,NaN,3.76,NaN,NaN,NaN,0.64,NaN,NaN,NaN,9387725.0,NaN,NaN,NaN,3075.0,NaN,NaN,NaN,2106585.0,NaN,NaN,NaN,0.71,NaN,NaN,NaN,78.0,NaN,NaN,NaN,34.30,NaN,NaN,NaN
2012-06-01,5.43,5.51,NaN,NaN,NaN,1.985991,NaN,NaN,NaN,100.02006,NaN,NaN,NaN,3.76,NaN,NaN,NaN,0.36,NaN,NaN,NaN,9368144.0,NaN,NaN,NaN,3067.0,NaN,NaN,NaN,2142061.0,NaN,NaN,NaN,0.74,NaN,NaN,NaN,77.0,NaN,NaN,NaN,32.34,NaN,NaN,NaN
2012-07-01,5.42,5.43,5.41,NaN,NaN,2.049195,1.854835,NaN,NaN,100.74394,98.87951,NaN,NaN,3.71,3.76,NaN,NaN,0.08,0.64,NaN,NaN,9517799.0,9387725.0,NaN,NaN,3073.0,3075.0,NaN,NaN,2175135.0,2106585.0,NaN,NaN,0.64,0.71,NaN,NaN,76.0,78.0,NaN,NaN,31.58,34.30,NaN,NaN
2012-08-01,5.40,5.42,5.51,NaN,NaN,2.028736,1.985991,NaN,NaN,100.90465,100.02006,NaN,NaN,3.72,3.76,NaN,NaN,0.43,0.36,NaN,NaN,9572403.0,9368144.0,NaN,NaN,3089.0,3067.0,NaN,NaN,2192888.0,2142061.0,NaN,NaN,0.68,0.74,NaN,NaN,75.0,77.0,NaN,NaN,30.71,32.34,NaN,NaN
2012-09-01,5.41,5.40,5.43,NaN,NaN,2.029443,2.049195,NaN,NaN,101.24114,100.74394,NaN,NaN,3.75,3.71,NaN,NaN,0.41,0.08,NaN,NaN,9584401.0,9517799.0,NaN,NaN,3099.0,3073.0,NaN,NaN,2219866.0,2175135.0,NaN,NaN,0.69,0.64,NaN,NaN,73.0,76.0,NaN,NaN,29.84,31.58,NaN,NaN
2012-10-01,5.34,5.41,5.42,5.41,NaN,2.028079,2.028736,1.854835,NaN,100.83720,100.90465,98.87951,NaN,3.72,3.72,3.76,NaN,0.57,0.43,0.64,NaN,9595876.0,9572403.0,9387725.0,NaN,3093.0,3089.0,3075.0,NaN,2244474.0,2192888.0,2106585.0,NaN,0.54,0.68,0.71,NaN,71.0,75.0,78.0,NaN,29.78,30.71,34.30,NaN
2012-11-01,5.21,5.34,5.40,5.51,NaN,2.029845,2.029443,1.985991,NaN,101.66903,101.24114,100.02006,NaN,3.76,3.75,3.76,NaN,0.59,0.41,0.36,NaN,9613316.0,9584401.0,9368144.0,NaN,3089.0,3099.0,3067.0,NaN,2276286.0,2219866.0,2142061.0,NaN,0.61,0.69,0.74,NaN,69.0,73.0,77.0,NaN,29.05,29.84,32.34,NaN
2012-12-01,5.10,5.21,5.41,5.43,NaN,2.067750,2.028079,2.049195,NaN,101.53680,100.83720,100.74394,NaN,3.67,3.72,3.71,NaN,0.60,0.57,0.08,NaN,9579068.0,9595876.0,9517799.0,NaN,3087.0,3093.0,3073.0,NaN,2310841.0,2244474.0,2175135.0,NaN,0.55,0.54,0.64,NaN,68.0,71.0,76.0,NaN,28.47,29.78,31.58,NaN


Rows com qualquer NA em X: 12
Rows totais: 165


In [5]:

cols_needed = [TARGET] + feature_cols

df_model = df_fe[cols_needed].dropna().copy()

X_model = df_model[feature_cols].copy()
y_model = df_model[TARGET].copy()

print("Dataset modelável (após dropna):")
print("df_model:", df_model.shape)
print("X_model:", X_model.shape, "| y_model:", y_model.shape)
print("Período modelável:", df_model.index.min().date(), "->", df_model.index.max().date())

df_model.head()

Dataset modelável (após dropna):
df_model: (153, 45)
X_model: (153, 44) | y_model: (153,)
Período modelável: 2013-04-01 -> 2025-12-01


,inadimplencia_pf_total,inadimplencia_pf_total__lag1,inadimplencia_pf_total__lag3,inadimplencia_pf_total__lag6,inadimplencia_pf_total__lag12,cambio_ptax_venda__lag1,cambio_ptax_venda__lag3,cambio_ptax_venda__lag6,cambio_ptax_venda__lag12,ibc_br_dessaz__lag1,ibc_br_dessaz__lag3,ibc_br_dessaz__lag6,ibc_br_dessaz__lag12,inadimplencia_carteira_total__lag1,inadimplencia_carteira_total__lag3,inadimplencia_carteira_total__lag6,inadimplencia_carteira_total__lag12,ipca_variacao_mensal__lag1,ipca_variacao_mensal__lag3,ipca_variacao_mensal__lag6,ipca_variacao_mensal__lag12,pmc_volume_vendas_dessazonalizado__lag1,pmc_volume_vendas_dessazonalizado__lag3,pmc_volume_vendas_dessazonalizado__lag6,pmc_volume_vendas_dessazonalizado__lag12,rendimento_medio_real__lag1,rendimento_medio_real__lag3,rendimento_medio_real__lag6,rendimento_medio_real__lag12,saldo_carteira_credito_total__lag1,saldo_carteira_credito_total__lag3,saldo_carteira_credito_total__lag6,saldo_carteira_credito_total__lag12,selic_acumulada_mes__lag1,selic_acumulada_mes__lag3,selic_acumulada_mes__lag6,selic_acumulada_mes__lag12,taxa_desocupacao__lag1,taxa_desocupacao__lag3,taxa_desocupacao__lag6,taxa_desocupacao__lag12,taxa_juros_pf_total__lag1,taxa_juros_pf_total__lag3,taxa_juros_pf_total__lag6,taxa_juros_pf_total__lag12
data_mes,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-04-01,4.85,4.94,5.03,5.34,5.41,1.982840,2.031077,2.029845,1.854835,101.75244,101.73475,101.66903,98.87951,3.45,3.54,3.76,3.76,0.47,0.86,0.59,0.64,9670157.0,9597714.0,9613316.0,9387725.0,3128.0,3092.0,3089.0,3075.0,2427062.0,2366165.0,2276286.0,2106585.0,0.55,0.60,0.61,0.71,80.0,72.0,69.0,78.0,27.42,27.38,29.05,34.30
2013-05-01,4.88,4.85,4.98,5.21,5.51,2.002214,1.973250,2.067750,1.985991,102.83683,100.81851,101.53680,100.02006,3.47,3.51,3.67,3.76,0.55,0.60,0.60,0.36,9712874.0,9587372.0,9579068.0,9368144.0,3135.0,3110.0,3087.0,3067.0,2449147.0,2383597.0,2310841.0,2142061.0,0.61,0.49,0.55,0.74,79.0,78.0,68.0,77.0,27.21,27.96,28.47,32.34
2013-06-01,4.61,4.88,4.94,5.10,5.43,2.034843,1.982840,2.077835,2.049195,103.05339,101.75244,101.11541,100.74394,3.47,3.45,3.55,3.71,0.37,0.47,0.79,0.08,9768338.0,9670157.0,9624551.0,9517799.0,3137.0,3128.0,3079.0,3073.0,2487294.0,2427062.0,2368338.0,2175135.0,0.60,0.55,0.55,0.64,76.0,80.0,69.0,76.0,26.93,27.42,27.32,31.58
2013-07-01,4.56,4.61,4.85,5.03,5.42,2.172955,2.002214,2.031077,2.028736,102.74886,102.83683,101.73475,100.90465,3.25,3.47,3.54,3.72,0.26,0.55,0.86,0.43,9813923.0,9712874.0,9597714.0,9572403.0,3163.0,3135.0,3092.0,3089.0,2531576.0,2449147.0,2366165.0,2192888.0,0.61,0.61,0.60,0.68,75.0,79.0,72.0,75.0,27.06,27.21,27.38,30.71
2013-08-01,4.45,4.56,4.88,4.98,5.40,2.252170,2.034843,1.973250,2.029443,103.04265,103.05339,100.81851,101.24114,3.21,3.47,3.51,3.75,0.03,0.37,0.60,0.41,10060612.0,9768338.0,9587372.0,9584401.0,3183.0,3137.0,3110.0,3099.0,2545127.0,2487294.0,2383597.0,2219866.0,0.72,0.60,0.49,0.69,74.0,76.0,78.0,73.0,27.62,26.93,27.96,29.84


Criamos transformações das séries (Δ, % e médias móveis) para capturar mudanças recentes e suavizar ruído. Fazemos isso no dataset completo antes de aplicar lags e cortar NaNs.

In [6]:
df_trans = df.copy()

COLS = [
    "cambio_ptax_venda",
    "ibc_br_dessaz",
    "inadimplencia_carteira_total",
    "ipca_variacao_mensal",
    "pmc_volume_vendas_dessazonalizado",
    "rendimento_medio_real",
    "saldo_carteira_credito_total",
    "selic_acumulada_mes",
    "taxa_desocupacao",
    "taxa_juros_pf_total",
]

# 1) Diferenças (Δ1m e Δ3m)
for col in COLS:
    df_trans[f"{col}__d1"] = df_trans[col].diff(1)
    df_trans[f"{col}__d3"] = df_trans[col].diff(3)

# 2) Crescimento percentual (para séries em nível)
PCT_COLS = [
    "cambio_ptax_venda",
    "ibc_br_dessaz",
    "pmc_volume_vendas_dessazonalizado",
    "rendimento_medio_real",
    "saldo_carteira_credito_total",
]
for col in PCT_COLS:
    df_trans[f"{col}__pct1"] = df_trans[col].pct_change(1)
    df_trans[f"{col}__pct3"] = df_trans[col].pct_change(3)

# 3) Médias móveis (MA3 e MA6)
for col in COLS:
    df_trans[f"{col}__ma3"] = df_trans[col].rolling(3).mean()
    df_trans[f"{col}__ma6"] = df_trans[col].rolling(6).mean()

# Amostra: últimas linhas (pra você ver se faz sentido)
sample_cols = [c for c in df_trans.columns if any(s in c for s in ["__d1", "__pct1", "__ma3"])]
display(df_trans[sample_cols].tail(12))

print("Total de colunas criadas:", len(df_trans.columns) - len(df.columns))

,cambio_ptax_venda__d1,ibc_br_dessaz__d1,inadimplencia_carteira_total__d1,ipca_variacao_mensal__d1,pmc_volume_vendas_dessazonalizado__d1,rendimento_medio_real__d1,saldo_carteira_credito_total__d1,selic_acumulada_mes__d1,taxa_desocupacao__d1,taxa_juros_pf_total__d1,cambio_ptax_venda__pct1,ibc_br_dessaz__pct1,pmc_volume_vendas_dessazonalizado__pct1,rendimento_medio_real__pct1,saldo_carteira_credito_total__pct1,cambio_ptax_venda__ma3,ibc_br_dessaz__ma3,inadimplencia_carteira_total__ma3,ipca_variacao_mensal__ma3,pmc_volume_vendas_dessazonalizado__ma3,rendimento_medio_real__ma3,saldo_carteira_credito_total__ma3,selic_acumulada_mes__ma3,taxa_desocupacao__ma3,taxa_juros_pf_total__ma3
data_mes,,,,,,,,,,,,,,,,,,,,,,,,,
2025-01-01,-0.075256,1.41189,0.24,-0.36,8734.0,17.0,1678.0,0.08,3.0,0.94,-0.012343,0.013206,0.000819,0.004942,0.000260,5.975286,107.687910,3.093333,0.356667,1.065829e+07,3441.333333,6.430808e+06,0.910000,62.666667,33.336667
2025-02-01,-0.256128,0.72789,0.07,1.15,58148.0,13.0,54271.0,-0.02,3.0,1.18,-0.042534,0.006720,0.005448,0.003760,0.008395,5.961482,108.092750,3.133333,0.663333,1.069032e+07,3455.666667,6.481955e+06,0.976667,65.000000,34.090000
2025-03-01,-0.018834,0.82721,0.02,-0.75,79103.0,10.0,60837.0,-0.03,2.0,0.42,-0.003267,0.007586,0.007371,0.002882,0.009333,5.844743,109.081747,3.243333,0.676667,1.073898e+07,3469.000000,6.520884e+06,0.986667,67.666667,34.936667
2025-04-01,0.036864,0.52728,0.22,-0.13,-33688.0,-11.0,50939.0,0.10,-4.0,0.50,0.006415,0.004799,-0.003116,-0.003161,0.007742,5.765377,109.775873,3.346667,0.766667,1.077350e+07,3473.000000,6.576233e+06,1.003333,68.000000,35.636667
2025-05-01,-0.116289,-1.37252,0.04,-0.17,-31394.0,15.0,35523.0,0.08,-4.0,0.35,-0.020106,-0.012432,-0.002913,0.004324,0.005358,5.732624,109.769863,3.440000,0.416667,1.077818e+07,3477.666667,6.625332e+06,1.053333,66.000000,36.060000
2025-06-01,-0.120331,-0.18401,0.03,-0.02,-7980.0,34.0,37454.0,-0.04,-4.0,0.18,-0.021232,-0.001688,-0.000743,0.009759,0.005619,5.666039,109.426780,3.536667,0.310000,1.075382e+07,3490.333333,6.666638e+06,1.100000,62.000000,36.403333
2025-07-01,-0.018564,-0.52888,0.20,0.02,-20018.0,-2.0,28905.0,0.18,-2.0,-0.48,-0.003347,-0.004859,-0.001864,-0.000569,0.004312,5.580977,108.731643,3.626667,0.253333,1.073403e+07,3506.000000,6.700598e+06,1.173333,58.666667,36.420000
2025-08-01,-0.081567,0.37480,0.18,-0.37,16189.0,0.0,39798.0,-0.12,0.0,0.20,-0.014754,0.003460,0.001510,0.000000,0.005911,5.507490,108.618947,3.763333,0.130000,1.073009e+07,3516.666667,6.735984e+06,1.180000,56.666667,36.386667
2025-09-01,-0.079515,-0.14359,-0.04,0.59,-14472.0,11.0,78208.0,0.06,0.0,-0.07,-0.014598,-0.001321,-0.001348,0.003129,0.011548,5.447608,108.519723,3.876667,0.210000,1.072399e+07,3519.666667,6.784954e+06,1.220000,56.000000,36.270000


Total de colunas criadas: 50


Vamos reduzir o risco de overfitting mantendo um conjunto enxuto e interpretável de features. Em séries macro mensais com ~165 observações, criar muitas transformações e muitos lags pode inflar a dimensionalidade e capturar ruído ao invés de sinal.
Por isso, vamos selecionar transformações que fazem sentido econômico e são mais estáveis:

Para variáveis do tipo taxa (Selic, taxa de juros PF, desemprego e IPCA), usaremos o nível, a variação mensal (Δ1m) e a média móvel de 3 meses (MA3), pois inadimplência tende a responder à tendência e mudanças recentes.

Para variáveis em nível/índice (câmbio, IBC-Br, PMC, renda e saldo de crédito), usaremos crescimento mensal (pct_change 1m) e MA3, para capturar dinâmica relativa e suavizar ruído.
Em seguida, aplicaremos apenas lags de curto/médio prazo (1, 3 e 6 meses) para garantir que o modelo use somente informação disponível até t−1 e reduzir dimensionalidade.

In [7]:
TARGET = "inadimplencia_pf_total"

# 1) Definir grupos por tipo (taxas vs níveis/índices)
RATE_COLS = ["selic_acumulada_mes", "taxa_juros_pf_total", "taxa_desocupacao", "ipca_variacao_mensal"]
LEVEL_COLS = ["cambio_ptax_venda", "ibc_br_dessaz", "pmc_volume_vendas_dessazonalizado", "rendimento_medio_real", "saldo_carteira_credito_total"]

# 2) Garantir que as transformações existem em df_trans (criado na célula 4)
# - taxas: nível, d1, ma3
# - níveis: pct1, ma3
BASE_FEATURES = []

for c in RATE_COLS:
    BASE_FEATURES += [c, f"{c}__d1", f"{c}__ma3"]

for c in LEVEL_COLS:
    BASE_FEATURES += [f"{c}__pct1", f"{c}__ma3"]

# Remove duplicatas e mantém só colunas que realmente existem
BASE_FEATURES = [c for c in dict.fromkeys(BASE_FEATURES) if c in df_trans.columns]

print("N de features base (antes de lags):", len(BASE_FEATURES))
print("Features base:")
display(pd.Series(BASE_FEATURES, name="feature"))

# 3) Criar lags apenas dessas features (mais enxuto)
LAGS = [1, 3, 6]

df_fe = df_trans.copy()

# lags do alvo (autoregressivo)
for k in LAGS:
    df_fe[f"{TARGET}__lag{k}"] = df_fe[TARGET].shift(k)

# lags das features selecionadas
for col in BASE_FEATURES:
    for k in LAGS:
        df_fe[f"{col}__lag{k}"] = df_fe[col].shift(k)

feature_cols = [c for c in df_fe.columns if c.endswith(tuple([f"lag{k}" for k in LAGS]))]

# 4) Dataset modelável (dropna) e separação X/y
df_model = df_fe[[TARGET] + feature_cols].dropna().copy()
X_model = df_model[feature_cols]
y_model = df_model[TARGET]

print("Após lags + dropna:")
print("X_model shape:", X_model.shape, "| y_model shape:", y_model.shape)
print("Período modelável:", df_model.index.min(), "->", df_model.index.max())

df_model.head()

N de features base (antes de lags): 22
Features base:


0                         selic_acumulada_mes
1                     selic_acumulada_mes__d1
2                    selic_acumulada_mes__ma3
3                         taxa_juros_pf_total
4                     taxa_juros_pf_total__d1
5                    taxa_juros_pf_total__ma3
6                            taxa_desocupacao
7                        taxa_desocupacao__d1
8                       taxa_desocupacao__ma3
9                        ipca_variacao_mensal
10                   ipca_variacao_mensal__d1
11                  ipca_variacao_mensal__ma3
12                    cambio_ptax_venda__pct1
13                     cambio_ptax_venda__ma3
14                        ibc_br_dessaz__pct1
15                         ibc_br_dessaz__ma3
16    pmc_volume_vendas_dessazonalizado__pct1
17     pmc_volume_vendas_dessazonalizado__ma3
18                rendimento_medio_real__pct1
19                 rendimento_medio_real__ma3
20         saldo_carteira_credito_total__pct1
21          saldo_carteira_credito

Após lags + dropna:
X_model shape: (157, 69) | y_model shape: (157,)
Período modelável: 2012-12-01 00:00:00 -> 2025-12-01 00:00:00


,inadimplencia_pf_total,inadimplencia_pf_total__lag1,inadimplencia_pf_total__lag3,inadimplencia_pf_total__lag6,selic_acumulada_mes__lag1,selic_acumulada_mes__lag3,selic_acumulada_mes__lag6,selic_acumulada_mes__d1__lag1,selic_acumulada_mes__d1__lag3,selic_acumulada_mes__d1__lag6,selic_acumulada_mes__ma3__lag1,selic_acumulada_mes__ma3__lag3,selic_acumulada_mes__ma3__lag6,taxa_juros_pf_total__lag1,taxa_juros_pf_total__lag3,taxa_juros_pf_total__lag6,taxa_juros_pf_total__d1__lag1,taxa_juros_pf_total__d1__lag3,taxa_juros_pf_total__d1__lag6,taxa_juros_pf_total__ma3__lag1,taxa_juros_pf_total__ma3__lag3,taxa_juros_pf_total__ma3__lag6,taxa_desocupacao__lag1,taxa_desocupacao__lag3,taxa_desocupacao__lag6,taxa_desocupacao__d1__lag1,taxa_desocupacao__d1__lag3,taxa_desocupacao__d1__lag6,taxa_desocupacao__ma3__lag1,taxa_desocupacao__ma3__lag3,taxa_desocupacao__ma3__lag6,ipca_variacao_mensal__lag1,ipca_variacao_mensal__lag3,ipca_variacao_mensal__lag6,ipca_variacao_mensal__d1__lag1,ipca_variacao_mensal__d1__lag3,ipca_variacao_mensal__d1__lag6,ipca_variacao_mensal__ma3__lag1,ipca_variacao_mensal__ma3__lag3,ipca_variacao_mensal__ma3__lag6,cambio_ptax_venda__pct1__lag1,cambio_ptax_venda__pct1__lag3,cambio_ptax_venda__pct1__lag6,cambio_ptax_venda__ma3__lag1,cambio_ptax_venda__ma3__lag3,cambio_ptax_venda__ma3__lag6,ibc_br_dessaz__pct1__lag1,ibc_br_dessaz__pct1__lag3,ibc_br_dessaz__pct1__lag6,ibc_br_dessaz__ma3__lag1,ibc_br_dessaz__ma3__lag3,ibc_br_dessaz__ma3__lag6,pmc_volume_vendas_dessazonalizado__pct1__lag1,pmc_volume_vendas_dessazonalizado__pct1__lag3,pmc_volume_vendas_dessazonalizado__pct1__lag6,pmc_volume_vendas_dessazonalizado__ma3__lag1,pmc_volume_vendas_dessazonalizado__ma3__lag3,pmc_volume_vendas_dessazonalizado__ma3__lag6,rendimento_medio_real__pct1__lag1,rendimento_medio_real__pct1__lag3,rendimento_medio_real__pct1__lag6,rendimento_medio_real__ma3__lag1,rendimento_medio_real__ma3__lag3,rendimento_medio_real__ma3__lag6,saldo_carteira_credito_total__pct1__lag1,saldo_carteira_credito_total__pct1__lag3,saldo_carteira_credito_total__pct1__lag6,saldo_carteira_credito_total__ma3__lag1,saldo_carteira_credito_total__ma3__lag3,saldo_carteira_credito_total__ma3__lag6
data_mes,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2012-12-01,5.10,5.21,5.41,5.43,0.55,0.54,0.64,-0.06,-0.15,-0.10,0.566667,0.636667,0.696667,28.47,29.78,31.58,-0.58,-0.06,-0.76,29.100000,30.110000,32.740000,68.0,71.0,76.0,-1.0,-2.0,-1.0,69.333333,73.000000,77.000000,0.60,0.57,0.08,0.01,0.16,-0.28,0.586667,0.470000,0.360000,0.018674,-0.000672,0.031825,2.041891,2.028753,1.963340,-0.001301,-0.003990,0.007237,101.347677,100.994330,99.881170,-0.003563,0.001197,0.015975,9.596087e+06,9.584227e+06,9.424556e+06,-0.000647,-0.001936,0.001956,3089.666667,3093.666667,3071.666667,0.015180,0.011085,0.015440,2.277200e+06,2.219076e+06,2.141260e+06
2013-01-01,5.03,5.10,5.34,5.42,0.55,0.61,0.68,0.00,0.07,0.04,0.570000,0.613333,0.686667,27.32,29.05,30.71,-1.15,-0.73,-0.87,28.280000,29.556667,31.543333,69.0,69.0,75.0,1.0,-2.0,-1.0,68.666667,71.000000,76.000000,0.79,0.59,0.43,0.19,0.02,0.35,0.660000,0.523333,0.290000,0.004877,0.000871,-0.009984,2.058477,2.029123,2.021307,-0.004150,0.008249,0.001595,101.440413,101.249123,100.556217,0.004748,0.001817,0.005737,9.605645e+06,9.597864e+06,9.486115e+06,-0.002592,-0.001293,0.005207,3085.000000,3093.666667,3076.333333,0.024881,0.014173,0.008162,2.318488e+06,2.246875e+06,2.170028e+06
2013-02-01,4.98,5.03,5.21,5.40,0.60,0.55,0.69,0.05,-0.06,0.01,0.566667,0.566667,0.670000,27.38,28.47,29.84,0.06,-0.58,-0.87,27.723333,29.100000,30.710000,72.0,68.0,73.0,3.0,-1.0,-2.0,69.666667,69.333333,74.666667,0.86,0.60,0.41,0.07,0.01,-0.02,0.750000,0.586667,0.306667,-0.022503,0.018674,0.000349,2.058887,2.041891,2.035792,0.006125,-0.001301,0.003335,101.462320,101.347677,100.963243,-0.002788,-0.003563,0.001253,9.600444e+06,9.596087e+06,9.558201e+06,0.004222,-0.000647,0.003237,3086.000000,3089.666667,3087.000000,-0.000918,0.015180,0.012302,2.348448e+06,2.277

In [8]:
df_model.to_parquet("../data/processed/base_modelagem_1.parquet", index=True)

# Base modelagem com implicações sugeridas na etapa de EDA

Nesta etapa vamos construir uma segunda versão do dataset de modelagem baseada nas conclusões da EDA. O objetivo é reduzir dimensionalidade e risco de overfitting usando (i) transformações que evitam tendência/regressão espúria (ex.: log no saldo de crédito e variações para séries em nível) e (ii) um único lag “ótimo” por variável (obtido via CCF), em vez de um grid de lags. Também incluiremos um componente autorregressivo do alvo (lag1), que se mostrou dominante, e manteremos todas as features defasadas para garantir ausência de data leakage (usar apenas informação até t−1).

In [1]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/base_unica_mensal.parquet").copy()

df["data_mes"] = pd.to_datetime(df["data_mes"])
df = df.sort_values("data_mes").set_index("data_mes")
df.index.name = "data_mes"

TARGET = "inadimplencia_pf_total"

# --------
# Transformações base (antes de lags)
# --------
df["saldo_credito_log"] = np.log(df["saldo_carteira_credito_total"])
df["cambio_pct1"] = df["cambio_ptax_venda"].pct_change(1)
df["desemprego_d1"] = df["taxa_desocupacao"].diff(1)
df["juros_pf_d3"] = df["taxa_juros_pf_total"].diff(3)

# --------
# Lags ótimos (CCF) — um lag por variável
# --------
LAG_MAP = {
    "y_lag1": (TARGET, 1),
    "selic_lag8": ("selic_acumulada_mes", 8),
    "desemprego_lag9": ("desemprego_d1", 9),
    "ibc_lag12": ("ibc_br_dessaz", 12),
    "juros_pf_lag6": ("juros_pf_d3", 6),
    "saldo_log_lag1": ("saldo_credito_log", 1),
    "inad_total_lag1": ("inadimplencia_carteira_total", 1),
    "cambio_lag1": ("cambio_pct1", 1),
}

for new_name, (col, k) in LAG_MAP.items():
    df[new_name] = df[col].shift(k)

FEATURES = list(LAG_MAP.keys())
df_model = df[[TARGET] + FEATURES].dropna().copy()

print("df_model shape:", df_model.shape)
print("Período modelável:", df_model.index.min(), "->", df_model.index.max())
df_model.head()

df_model shape: (153, 9)
Período modelável: 2013-04-01 00:00:00 -> 2025-12-01 00:00:00


,inadimplencia_pf_total,y_lag1,selic_lag8,desemprego_lag9,ibc_lag12,juros_pf_lag6,saldo_log_lag1,inad_total_lag1,cambio_lag1
data_mes,,,,,,,,,
2013-04-01,4.85,4.94,0.69,-1.0,98.87951,-1.66,14.702192,3.45,0.004860
2013-05-01,4.88,4.85,0.54,-2.0,100.02006,-1.37,14.711250,3.47,0.009771
2013-06-01,4.61,4.88,0.61,-2.0,100.74394,-2.46,14.726706,3.47,0.016297
2013-07-01,4.56,4.61,0.55,-2.0,100.90465,-1.67,14.744353,3.25,0.067874
2013-08-01,4.45,4.56,0.55,-1.0,101.24114,-0.51,14.749691,3.21,0.036455


In [2]:
df_model.to_parquet("../data/processed/base_modelagem_2.parquet", index=True)